# 🛠️ Notebook 2: LinkedIn (professional network) — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/linkedin
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from itertools import count

_uid = count(1); _jid = count(1); _rid = count(1); _aid = count(1)

@dataclass
class Experience:
    title: str
    company: str
    start_year: int
    end_year: int | None = None

@dataclass
class Profile:
    headline: str = ""
    experiences: list[Experience] = field(default_factory=list)
    education: list[str] = field(default_factory=list)
    skills: set[str] = field(default_factory=set)

@dataclass
class User:
    name: str
    id: int = field(default_factory=lambda: next(_uid))
    profile: Profile = field(default_factory=Profile)
    connections: set["User"] = field(default_factory=set)

    def __hash__(self): return self.id
    def __eq__(self, o): return isinstance(o, User) and self.id == o.id
    def __repr__(self): return f"User({self.name})"


In [ ]:
class ReqStatus(Enum):
    PENDING="pending"; ACCEPTED="accepted"; REJECTED="rejected"

@dataclass
class ConnectionRequest:
    sender: User
    receiver: User
    id: int = field(default_factory=lambda: next(_rid))
    status: ReqStatus = ReqStatus.PENDING

    def accept(self):
        if self.status != ReqStatus.PENDING:
            raise ValueError("already resolved")
        self.status = ReqStatus.ACCEPTED
        self.sender.connections.add(self.receiver)
        self.receiver.connections.add(self.sender)

    def reject(self):
        if self.status != ReqStatus.PENDING:
            raise ValueError("already resolved")
        self.status = ReqStatus.REJECTED


@dataclass
class Company:
    name: str
    employees: list[User] = field(default_factory=list)

@dataclass
class Job:
    title: str
    company: Company
    description: str
    required_skills: set[str] = field(default_factory=set)
    id: int = field(default_factory=lambda: next(_jid))

class AppStatus(Enum):
    SUBMITTED="submitted"; REVIEWED="reviewed"; OFFER="offer"; REJECTED="rejected"

@dataclass
class Application:
    user: User
    job: Job
    id: int = field(default_factory=lambda: next(_aid))
    status: AppStatus = AppStatus.SUBMITTED

    def match_score(self) -> float:
        if not self.job.required_skills: return 1.0
        matched = self.user.profile.skills & self.job.required_skills
        return len(matched) / len(self.job.required_skills)


In [ ]:
alice = User("Alice")
alice.profile.headline = "Backend engineer"
alice.profile.skills = {"python","sql","redis"}
alice.profile.experiences.append(Experience("Eng", "Foo Inc", 2020, 2024))

bob = User("Bob")
bob.profile.skills = {"python","go"}

# Connection request flow
req = ConnectionRequest(alice, bob)
print("before:", alice.connections, bob.connections)
req.accept()
print("after :", alice.connections, bob.connections)

# Jobs
foo = Company("Foo Inc", employees=[alice])
job = Job("Senior Backend", foo, "Distributed systems",
          required_skills={"python","sql","kafka"})

a1 = Application(alice, job); a2 = Application(bob, job)
print("Alice match score:", a1.match_score())  # 2/3
print("Bob   match score:", a2.match_score())  # 1/3

# Business rule: can't connect twice with pending request outstanding
try:
    ConnectionRequest(alice, bob).accept()
    # already connected but status flow still works — in real system you'd validate beforehand
except Exception as e:
    print(e)


### Try it
- Add a `Message`/`Inbox` so connected users can DM.
- Add `Endorsement`: users endorse each other's skills.
- Add a `JobRecommender` ranking jobs by `match_score` and connection-in-company signals.